In [3]:
import pickle 
import os
from pathlib import Path
from scipy import stats
import numpy as np
import datetime
import numpy as np
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import matplotlib.pyplot as plt


### Modelling Phase 2

| Commodity | ARMA (p, q) | GJR-GARCH (p, o, q) |
| :--- | :--- | :--- |
| **NG** | 1, 0 | 1, 0, 1 |
| **Copper** | 0, 0 | 1, 1, 1 |
| **Brent** | 0, 0 | 1, 1, 1 |
| **Corn** | 0, 0 | 1, 1, 1 |
| **WTI** | 3, 3 | 1, 1, 1 |
| **Cattle** | 0, 0 | 1, 0, 1 |
| **Cocoa** | 0, 0 | 1, 0, 1 |

*Note: The `o` parameter represents the asymmetry indicator.*

In [1]:
%pwd

'c:\\Users\\BudgetGamer\\Projects\\Ian\\CommodityProject\\SystemicRiskCopulaProject\\Modelling'

In [4]:
current_path = Path.cwd()
try:
    project_root = next(p for p in current_path.parents if p.name == "CommodityProject")
except StopIteration:
    print(Path.cwd())
except e as exception:
    print(e)

In [16]:
%cd $project_root   

c:\Users\BudgetGamer\Projects\Ian\CommodityProject


In [17]:
data_dir = os.path.join(project_root, "SystemicRiskCopulaProject", "Data")

### Reading data

In [19]:
%cd $data_dir

c:\Users\BudgetGamer\Projects\Ian\CommodityProject\SystemicRiskCopulaProject\Data


In [43]:
file_list = os.listdir(data_dir)

for file in file_list:
    if file.endswith(".pkl"):
        _file = Path(file).stem
        with open(os.path.join(data_dir, file), "rb") as f:
            globals()[_file] = pickle.load(f)
        print(f"Loaded {file} with keys: {list(globals()[_file].keys())}")
        

Loaded arch_dict.pkl with keys: ['NG', 'Copper', 'Brent', 'Corn', 'WTI', 'Cattle', 'Cocoa']
Loaded arma_dict.pkl with keys: ['NG', 'Copper', 'Brent', 'Corn', 'WTI', 'Cattle', 'Cocoa']
Loaded cleaned_dfs.pkl with keys: ['NG', 'Copper', 'Brent', 'Corn', 'WTI', 'Cattle', 'Cocoa']
Loaded dfs.pkl with keys: ['NG', 'Copper', 'Brent', 'Corn', 'WTI', 'Cattle', 'Cocoa']


### Starting Copula maths

Now we have the conditional mean and conditional variance, and also the standardized residuals, $\eta$, we can start the Copula math. ARMA will be able to generate the mean and standard residual along iwth condiotional variance would be able to generate variance involved. The only source of randomness or stochastic nature is shock term, $\eta$. <br> <br>

$$R_{i,t} = \mu_{i,t} + \sigma_{i,t} \cdot \eta_{i,t} 




$\mu_{i,t}$ :  deterministic - based on yesterdays return, R_{i,t-1} <br>
$\sigma$ : deterministic - based on yesterdays variance <br>
$\eta$ : Standardized shock - Random component <br>

### How the Co-Simulation Actually Unfolds (Step-by-Step)

Suppose you are simulating tomorrow ($t = T+1$) for $C_1$ (Brent) and $C_2$ (WTI).

#### Step A: Determine the deterministic environment

    Before tomorrow's trading bell rings, you already know today's closing prices.
        * You calculate $\mu_{1, T+1}$ and $\mu_{2, T+1}$ using their respective ARMA equations.
        * You calculate $\sigma_{1, T+1}$ and $\sigma_{2, T+1}$ using their respective GARCH equations.
                (Notice: these numbers are fixed. No random number generator is called here.)
#### Step B: Call the Copula for the Joint Shock
    To see what unexpected shock hits the market tomorrow, you sample from the fitted multivariate Copula:
        * The Copula draws correlated uniform numbers:$$(u_1, u_2)  ~ \text{Copula}$$
        
            (If Brent plunges in the copula draw, $u_1 = 0.01$; because of tail dependence, $u_2$ is automatically drawn as $0.012$.)
        * Invert each $u_i$ back through its individual Skewed-$t$ CDF (the inverse Probability Integral Transform, $F_i^{-1}$):
        
            $$\eta_1 = F_{\text{Skew-}t, 1}^{-1}(u_1) = -3.8\sigma$$
            $$\eta_2 = F_{\text{Skew-}t, 2}^{-1}(u_2) = -3.5\sigma$$
                                (-3.8 = F_-1(0.01) and -3.5 = F-1) Those numbers (-3.8 and -3.5) were hypothetical examples I used to illustrate what a "joint crash" looks like mathematically.
                We are transforming CDF(0,1) back to real world shock here. 
                The shocks $(\eta_1, \eta_2)$ now contain all the non-linear tail dependence, joint crash correlation, and cross-asset linkage.

#### Step C: Synthesize Tomorrow's Return
    Now, scale and shift the shocks using the pre-computed deterministic volatility and mean:
    
    $$R_{1, T+1} = \mu_{1, T+1} + \sigma_{1, T+1} \cdot \eta_1$$
    $$R_{2, T+1} = \mu_{2, T+1} + \sigma_{2, T+1} \cdot \eta_2$$
    
    Because $\eta_1$ and $\eta_2$ plummeted together in the simulation, $R_1$ and $R_2$ plunge together.
    
#### What Happens at Step $t = T+2$ (Dynamic Feedback)?

    This is where the magic happens over multi-day simulations.
    When you advance to the next day ($T+2$):
        
        $C_1$'s new conditional volatility $\sigma_{1, T+2}$ is calculated using $\epsilon_{1, T+1} = \sigma_{1, T+1} \cdot \eta_1$.
        
        $C_2$'s new conditional volatility $\sigma_{2, T+2}$ is calculated using $\epsilon_{2, T+1} = \sigma_{2, T+1} \cdot \eta_2$.
        
    Because both assets took a joint crash in step $T+1$ through the copula shocks ($\eta_1, \eta_2$), both assets' GARCH models automatically spike their volatility forecasts simultaneously for step $T+2$.